In [ ]:

import os
from getpass import getpass
from openai import OpenAI

# Enter your Groq API key securely
if not os.environ.get("GROQ_API_KEY"):
    GROQ_API_KEY = getpass("Enter your Groq API key: ").strip()
    if not GROQ_API_KEY:
        raise ValueError("API key cannot be empty.")
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
else:
    GROQ_API_KEY = os.environ["GROQ_API_KEY"]

# Connect to Groq
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    timeout=30.0,
    max_retries=1
)

print("Groq client configured successfully!")

Enter your Groq API key: ··········
Groq client configured successfully!


In [ ]:

GROQ_MODEL = "openai/gpt-oss-120b"

try:
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are the AI engine powering an intelligent project-planning system."
            },
            {
                "role": "user",
                "content": "Reply with exactly: AI ENGINE READY"
            }
        ],
        temperature=0,
        max_tokens=50
    )

    print("Model:", GROQ_MODEL)
    print("Response:", response.choices[0].message.content)
    print("LLM connection test successful!")

except Exception as e:
    print("Connection test failed:", type(e).__name__)
    print("Details:", str(e))

Model: openai/gpt-oss-120b
Response: AI ENGINE READY
LLM connection test successful!


In [ ]:

import json
from IPython.display import display, Markdown

# Store the workflow history for our final report
workflow_history = []

def run_ai_role(role, objective, context):
    """Run one specialized AI role and record its result."""

    print(f"\nRunning AI role: {role}...")

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    f"You are {role} in an AI Idea-to-Execution Engine. "
                    "Be practical, structured, honest about uncertainty, "
                    "and specific. Do not invent facts or claim to have "
                    "verified information that you have not checked."
                )
            },
            {
                "role": "user",
                "content": (
                    f"OBJECTIVE:\n{objective}\n\n"
                    f"CONTEXT:\n{context}\n\n"
                    "Provide a clear, useful response with headings and "
                    "actionable recommendations."
                )
            }
        ],
        temperature=0.2,
        max_tokens=1800
    )

    result = response.choices[0].message.content or ""

    if not result.strip():
        raise ValueError(f"{role} returned an empty response.")

    workflow_history.append({
        "role": role,
        "objective": objective,
        "result": result
    })

    print(f"Completed: {role}")
    return result

print("AI agent engine created successfully!")

AI agent engine created successfully!


In [ ]:

def retrieve_documents(question, top_k=3):
    query_vector = rag_vectorizer.transform([question])

    similarities = cosine_similarity(
        query_vector, rag_matrix
    ).flatten()

    top_indices = similarities.argsort()[::-1][:top_k]
    results = []

    for idx in top_indices:
        score = float(similarities[idx])

        if score > 0:
            results.append({
                **chunks[idx],
                "score": round(score, 3)
            })

    return results


question = input("Ask a question about AI: ")
results = retrieve_documents(question)

if results:
    for i, item in enumerate(results, start=1):
        print(f"\n--- Retrieved result {i} ---")
        print("Topic:", item["topic"])
        print("Relevance score:", item["score"])
        print("Content:", item["text"])
else:
    print("No matching information found in the knowledge base.")

Ask a question about AI: How does RAG work?

--- Retrieved result 1 ---
Topic: Retrieval-Augmented Generation
Relevance score: 0.25
Content: Retrieval-Augmented Generation (RAG) combines information retrieval with language generation. First, a system searches a knowledge base for relevant text. Then it provides that text as context to an LLM, which generates an answer grounded in the retrieved material. RAG can improve the usefulness of answers about specific documents, but it does not guarantee that every answer is correct.


In [ ]:

def rag_answer(question):
    """Answer a question using retrieved knowledge and the Groq LLM."""

    retrieved = retrieve_documents(question, top_k=3)

    if not retrieved:
        return (
            "I could not find relevant information in the current "
            "knowledge base. Please add relevant notes or ask a "
            "question related to the available topics."
        )

    context = "\n\n".join(
        f"Topic: {item['topic']}\nContent: {item['text']}"
        for item in retrieved
    )

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful AI study assistant. Answer the "
                    "user's question using the supplied retrieved notes. "
                    "Do not invent facts or claim the notes say something "
                    "they do not. If the notes do not contain enough "
                    "information, clearly state that limitation. "
                    "Keep the answer clear and student-friendly."
                )
            },
            {
                "role": "user",
                "content": (
                    f"QUESTION:\n{question}\n\n"
                    f"RETRIEVED NOTES:\n{context}\n\n"
                    "Answer using the retrieved notes."
                )
            }
        ],
        temperature=0.2,
        max_tokens=600
    )

    answer = response.choices[0].message.content or ""

    print("\nRetrieved topics:")
    for item in retrieved:
        print(f"- {item['topic']} (score: {item['score']})")

    return answer


# Test the complete retrieval + generation pipeline.
question = input("Ask your AI study assistant: ")
answer = rag_answer(question)

display(Markdown("## AI Study Assistant Answer\n" + answer))

Ask your AI study assistant: How can an AI study assistant identify weak topics?

Retrieved topics:
- AI Study Assistant (score: 0.373)
- Large Language Models (score: 0.036)
- Responsible AI (score: 0.03)


## AI Study Assistant Answer
**How an AI‑powered study assistant can spot the topics you’re struggling with**

1. **Collect performance data**  
   * The assistant gives you short quizzes or practice questions for each subject area.  
   * Every answer is recorded, so the system knows how many points you earned on each topic.

2. **Turn scores into a “mastery estimate”**  
   * For each topic the assistant calculates an average score (e.g., 85 % on algebra, 62 % on chemistry).  
   * Because quiz scores are only an *approximate* measure of understanding, the assistant treats them as a **signal, not a final verdict**.

3. **Identify the low‑scoring topics**  
   * Topics whose average score falls below a chosen threshold (say, 70 %) are flagged as “weak.”  
   * The assistant can also look for patterns—e.g., a steady decline over several quizzes or many missed questions of the same type.

4. **Prioritize revision time**  
   * Once weak topics are identified, the assistant automatically schedules extra review sessions for them.  
   * It may generate extra practice questions, concise explanations, or short video summaries focused on those areas.

5. **Give you feedback you can act on**  
   * The assistant shows a simple dashboard: “You’re doing well in Geometry (90 %), but need more work on Fractions (58 %).”  
   * It also explains that the scores are an *approximation* and encourages you to double‑check any concepts you’re still unsure about.

---

### Why this works (and its limits)

- **Why scores help:** Quiz results give a concrete, quantifiable snapshot of what you got right or wrong, making it easy for the AI to compare topics side‑by‑side.  
- **Why they’re only approximate:** A single quiz might be affected by test anxiety, a typo, or a lucky guess. The assistant therefore treats the scores as a *starting point* and may ask for additional data (more quizzes, self‑ratings, or time‑on‑task) to refine its picture of your mastery.

---

### Quick checklist for students

| Step | What you (or the assistant) does |
|------|-----------------------------------|
| 1️⃣   | Take short quizzes for each topic. |
| 2️⃣   | The assistant calculates average scores per topic. |
| 

In [ ]:

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# General-purpose knowledge for the AI Idea-to-Execution Engine.
knowledge_base = [
    {
        "source": "Project Planning Guide",
        "topic": "Idea Analysis",
        "text": """
        A good idea analysis identifies the problem being solved,
        target users, proposed solution, unique value, assumptions,
        and expected benefits. Ideas should be described clearly.
        The first version should focus on a specific problem and
        a small set of essential features.
        """
    },
    {
        "source": "Project Planning Guide",
        "topic": "Feasibility Evaluation",
        "text": """
        Feasibility evaluation considers technical feasibility,
        time, budget, available skills, resources, and operational
        constraints. A prototype should use tools and services
        that are realistically available. Estimates are assumptions,
        not guarantees, and should be validated during development.
        """
    },
    {
        "source": "Project Planning Guide",
        "topic": "Execution Roadmap",
        "text": """
        An execution roadmap breaks an idea into ordered tasks.
        Each task should have a priority, estimated duration,
        dependencies, deliverable, and acceptance criteria.
        Start with the minimum viable product, test the core
        functionality, and add optional features after the basic
        workflow works reliably.
        """
    },
    {
        "source": "Project Planning Guide",
        "topic": "Risk Management",
        "text": """
        Common project risks include technical failures, unclear
        requirements, unrealistic schedules, unexpected costs,
        unreliable data, and external service limits. For each risk,
        estimate its likelihood and impact, then define a mitigation.
        Keep a fallback approach for important dependencies.
        """
    },
    {
        "source": "Project Planning Guide",
        "topic": "Testing and Validation",
        "text": """
        Testing checks whether each feature behaves as expected.
        Define measurable acceptance criteria before testing.
        Test normal inputs, empty inputs, unusual inputs, and failures.
        Validate important outputs rather than assuming generated
        content is correct. Record limitations and unresolved issues.
        """
    },
    {
        "source": "Project Planning Guide",
        "topic": "AI and RAG",
        "text": """
        A large language model can analyze ideas, explain options,
        generate plans, and review drafts. Retrieval-Augmented
        Generation retrieves relevant information from a knowledge
        base and supplies it as context to the model. Answers should
        distinguish retrieved evidence from assumptions and general
        suggestions. Retrieval does not guarantee correctness.
        """
    },
    {
        "source": "Project Planning Guide",
        "topic": "Agentic Workflow",
        "text": """
        An agentic workflow divides a complex task into specialized
        stages. For example, an analyst examines an idea, an evaluator
        checks feasibility, a planner creates milestones, and a critic
        reviews the plan. Feedback can trigger a limited revision cycle.
        The workflow should have clear responsibilities and a maximum
        number of revision attempts.
        """
    },
    {
        "source": "Project Planning Guide",
        "topic": "Minimum Viable Product",
        "text": """
        A minimum viable product (MVP) is the smallest useful version
        that demonstrates the main value of an idea. Prioritize core
        functionality, avoid unnecessary infrastructure, and postpone
        advanced features until the prototype works. A student MVP
        should be achievable with the available time, skills, and budget.
        """
    }
]

# Rebuild the searchable chunks and TF-IDF index.
chunks = [
    {
        "source": doc["source"],
        "topic": doc["topic"],
        "text": re.sub(r"\s+", " ", doc["text"]).strip()
    }
    for doc in knowledge_base
]

rag_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

rag_matrix = rag_vectorizer.fit_transform(
    [chunk["text"] for chunk in chunks]
)

print("General-purpose knowledge base ready!")
print("Project: AI Idea-to-Execution Engine")
print("Knowledge topics:", len(chunks))
print("Study-assistant-specific content removed.")

General-purpose knowledge base ready!
Project: AI Idea-to-Execution Engine
Knowledge topics: 8
Study-assistant-specific content removed.


In [ ]:

def rag_answer(question):
    """Answer project-planning questions using RAG and Groq."""

    retrieved = retrieve_documents(question, top_k=3)

    if not retrieved:
        return (
            "I couldn't find relevant information in the project "
            "knowledge base. Try asking about idea analysis, "
            "feasibility, execution roadmaps, risks, testing, "
            "MVPs, or agentic workflows."
        )

    context = "\n\n".join(
        f"Topic: {item['topic']}\n"
        f"Source: {item['source']}\n"
        f"Information: {item['text']}"
        for item in retrieved
    )

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are the AI Idea-to-Execution Engine. "
                    "Help users turn ideas into practical action plans. "
                    "You can analyze ideas, evaluate feasibility, "
                    "identify risks, recommend an MVP, and plan tasks. "
                    "Use the retrieved knowledge when relevant. "
                    "Do not pretend that general guidance is verified "
                    "evidence. If the retrieved notes are insufficient, "
                    "say so and label any additional advice as a general "
                    "recommendation. Do not assume every idea is about "
                    "education or study assistance."
                )
            },
            {
                "role": "user",
                "content": (
                    f"USER QUESTION:\n{question}\n\n"
                    f"RETRIEVED PROJECT KNOWLEDGE:\n{context}\n\n"
                    "Give a clear, practical answer with actionable steps."
                )
            }
        ],
        temperature=0.2,
        max_tokens=700
    )

    answer = response.choices[0].message.content or ""

    print("\nKnowledge retrieved:")
    for item in retrieved:
        print(f"- {item['topic']} (relevance: {item['score']})")

    return answer


# Test the project-specific RAG assistant.
question = input("Ask the AI Idea-to-Execution Engine: ")
answer = rag_answer(question)

display(Markdown("## AI Idea-to-Execution Engine\n\n" + answer))

Ask the AI Idea-to-Execution Engine: I want to build a smart agriculture system. How should I start and evaluate its feasibility?

Knowledge retrieved:
- Feasibility Evaluation (relevance: 0.161)
- Execution Roadmap (relevance: 0.109)
- Agentic Workflow (relevance: 0.067)


## AI Idea-to-Execution Engine

Below is a **step‑by‑step playbook** that follows the three‑stage “agentic workflow” (analyst → evaluator → planner → critic) and incorporates the **Feasibility Evaluation** and **Execution Roadmap** guidelines you already have.

---

## 1️⃣ ANALYST – Clarify the Idea & Scope

| Question | What to decide / document |
|----------|----------------------------|
| **Problem statement** | “Small‑holder farmers need real‑time data on soil moisture, temperature, and pest pressure to reduce water use and increase yields.” |
| **Target users** | • 5–20 ha farms in a specific region (e.g., Midwest corn growers) <br>• Extension agents who will view dashboards |
| **Core value proposition** | • Automated irrigation scheduling <br>• Early‑warning pest alerts |
| **Key data sources** | • Soil‑moisture sensors (capacitive) <br>• Weather API (e.g., OpenWeather) <br>• Camera‑based pest detection (optional) |
| **Success metric for MVP** | Reduce irrigation water use by ≥ 15 % on pilot fields while maintaining yields. |
| **Constraints you already know** | • Limited budget (< $15 k for pilot) <br>• No in‑house hardware design team <br>• Internet connectivity is intermittent in the field |

*Deliverable:* **One‑page Idea Brief** (problem, users, value, data, success metric, constraints). Keep it short – it will be the reference for the next stage.

---

## 2️⃣ EVALUATOR – Feasibility Check (use the “Feasibility Evaluation” checklist)

| Dimension | Evaluation Questions | Preliminary Answer (you can adjust) | Risk / Mitigation |
|-----------|----------------------|--------------------------------------|-------------------|
| **Technical** | • Are the sensors commercially available? <br>• Can they talk to a gateway with LTE/LoRa? <br>• Is cloud storage / analytics affordable? | Yes – cheap capacitive moisture probes ($10 each) + LoRaWAN gateway ($200). Cloud (AWS IoT Core) free tier for pilot. | *Risk*: LoRa coverage gaps → **Mitigation**: Use cellular gateway as fallback. |
| **Time** | • How long to prototype hardware + software? | 6 weeks (2 w for hardware procurement & integration, 4 w for cloud pipeline). | *Risk*: Integration delays → **Mitigation**: Use pre‑built sensor kits (e.g., Seeed Studio). |
| **Budget** | • Total cost of hardware, cloud, labor? | Sensors (10 × $10) = $100 <br>Gateway = $200 <br>Cloud (first 3 months) ≈ $50 <br>Developer time (2 person‑weeks) ≈ $2

In [ ]:

import time

planner_objective = """
Create a concise execution roadmap for the AI Idea-to-Execution Engine's
current submitted idea.

Include:
1. Project objective
2. Five to seven prioritized implementation steps
3. Approximate duration and deliverable for each step
4. Required tools and dependencies
5. Three testing criteria
6. Three major risks and mitigations
7. Features to postpone

Constraints:
- Student-level prototype using Python and Google Colab.
- Prefer simple, low-cost tools.
- Separate prototype features from future hardware deployment.
- Label estimates as estimates.
- Do not invent verified prices, statistics, or results.
- Keep the entire response under 900 words.
"""

print("Waiting briefly before retrying...")
time.sleep(12)

try:
    execution_plan = run_ai_role(
        role="Execution Planning Strategist",
        objective=planner_objective,
        context=(
            f"IDEA:\n{sample_idea}\n\n"
            f"ANALYSIS SUMMARY:\n{idea_analysis[:2500]}\n\n"
            f"FEASIBILITY SUMMARY:\n{feasibility_analysis[:2500]}"
        )
    )

    display(Markdown("## 3. Execution Roadmap\n\n" + execution_plan))

except Exception as e:
    print("Planner could not complete:", type(e).__name__)
    print(str(e))
    print("\nPlease wait 30–60 seconds before trying again.")

Waiting briefly before retrying...

Running AI role: Execution Planning Strategist...
Completed: Execution Planning Strategist


## 3. Execution Roadmap

**Smart‑Agriculture Irrigation Assistant – Student‑Level Prototype (Google Colab)**  

---

## 1. Project Objective  
Create a low‑cost, easy‑to‑use system that (a) gathers soil‑moisture and basic weather data, (b) runs a simple rule‑based model in a Colab notebook, and (c) delivers a one‑page irrigation recommendation (amount & timing) to the farmer via a free messaging channel. The prototype must be reproducible with off‑the‑shelf parts and free cloud services.

---

## 2. Prioritized Implementation Steps  

| # | Step (high‑level) | Approx. Duration* | Main Deliverable |
|---|-------------------|-------------------|------------------|
| 1 | **Define requirements & acquire hardware** – decide moisture‑thresholds, select sensor kit, create free API accounts (OpenWeatherMap, Telegram Bot, Google Sheets). | 1 wk (≈10 h) | Requirements doc + parts list + API keys |
| 2 | **Build & test sensor node** – wire capacitive moisture sensor + DHT22 (or rely on public weather), program ESP32/Arduino Nano to publish JSON to Google Sheets via HTTPS. | 1 wk (≈12 h) | Arduino/ESP sketch + verified data in a test Sheet |
| 3 | **Set up cloud data pipeline** – create a Google Sheet, share with a service account, write a small Python helper (using `gspread` or `gsheetsdb`) that can read the latest row. Verify write‑and‑read latency (< 30 s). | 1 wk (≈10 h) | Working read/write demo script |
| 4 | **Develop Colab notebook** –  <br>• Pull sensor data & weather forecast (OpenWeatherMap). <br>• Apply rule‑engine: `if moisture < thresh and ET₀ > X → water`. <br>• Plot recent 7‑day trends (Matplotlib/Plotly). <br>• Auto‑generate a PDF/HTML recommendation. | 2 wks (≈20 h) | Fully runnable notebook that outputs a recommendation page |
| 5 | **Integrate messaging** – create a Telegram bot (or Twilio WhatsApp sandbox), add a function in the notebook that sends the PDF/summary to a pre‑registered phone number. Test end‑to‑end delivery. | 1 wk (≈10 h) | Bot script + successful message to a test phone |
| 6 | **User‑testing & documentation** – run the system for 3‑5 days on a small plot, collect farmer feedback on clarity, timing, and perceived usefulness. Write a short user guide (setup, thresholds, how to read the recommendation). | 1 wk (≈8 h) | Field‑test report + README/step‑by‑step guide |
| 7 | **Wrap‑up & demo preparation** – package the notebook, code, and hardware schematics into a GitHub repo; prepare a 5‑minute Colab‑based demo for class or extension workshop. | 2 days (≈4 h) | Public repo + slide deck |

\*All durations are **estimates** for a motivated student team (2‑3 members) working part‑time.

---

## 3. Required Tools & Dependencies  

| Category | Tool / Service | Reason for Choice | Cost (estimate) |
|----------|----------------|-------------------|-----------------|
| **Hardware** | ESP32 dev board, capacitive soil‑moisture sensor, DHT22 (optional) | Wi‑Fi built‑in, cheap, Arduino‑compatible | <$30 total |
| **Cloud storage** | Google Sheets (free tier) + Google Service Account | Simple CSV‑like DB, easy `gspread` access | Free |
| **Weather data** | OpenWeatherMap Current & 5‑day forecast (free tier) | No cost, sufficient for demo | Free |
| **Messaging** | Telegram Bot (free) **or** Twilio WhatsApp sandbox (free trial) | Instant delivery, no SMS fees for demo | Free |
| **Development** | Google Colab (free), Python 3, libraries: `requests`, `gspread`, `pandas`, `matplotlib`/`plotly`, `fpdf` or `pdfkit` | All run in the notebook, no local install needed | Free |
| **Version control** | GitHub (public repo) | Collaboration & reproducibility | Free |
| **Documentation** | Markdown + Jupyter‑style cells | Inline explanations for students | Free |

---

## 4. Testing Criteria  

1. **Data Integrity** – The latest sensor row in Google Sheets matches the value read directly from the ESP32 (±5 % tolerance) for at least 95 % of samples over a 48‑hour period.  
2. **Decision Accuracy (Rule‑Engine)** – For a set of manually‑labelled “water‑needed” vs. “no‑water” days (based on ground‑truth moisture thresholds), the notebook’s recommendation matches the label ≥ 80 % of the time.  
3. **Message Delivery** – The PDF/summary is received on the farmer’s device within 2 minutes of notebook execution for 9 out of 10 test runs.

---

## 5. Major Risks & Mitigations  

| Risk | Why it matters | Mitigation |
|------|----------------|------------|
| **Connectivity gaps** – Rural field may have intermittent Wi‑Fi or cellular data, breaking sensor‑to‑cloud upload. | System stalls, farmer receives no advice. | Use ESP32’s built‑in Wi‑Fi retry logic; buffer readings locally (EEPROM) and push when connection restores. Provide an offline fallback: a simple “check soil manually” note in the guide. |
| **Threshold selection uncertainty** – Wrong moisture threshold leads to over‑ or under‑watering, eroding trust. | Farmers may ignore recommendations. | Start with literature‑based thresholds for the target crop; allow the user (extension agent) to adjust thresholds in the notebook via a simple UI cell. Collect feedback during the 3‑day field test to refine. |
| **Free‑service rate limits** – OpenWeatherMap or Telegram may throttle after many calls, especially during demo. | Notebook crashes or messages delayed. | Cache the weather response for the whole day (store in Sheet); limit notebook runs to once per day for the demo. If needed, switch to a second free API key. |

---

## 6. Features to Postpone (Future Phases)  

| Feature | Reason for postponement |
|---------|--------------------------|
| **Automated valve control** – Directly open/close irrigation pumps via relays. | Requires additional hardware safety review and power‑handling design; not needed for a recommendation‑only demo. |
| **Machine‑learning model** – Train a regression on historic yield vs. irrigation. | Adds data‑collection burden and complexity; rule‑based logic is sufficient to prove concept. |
| **Multi‑field dashboard** – Aggregate data from dozens of farms in a single UI. | Needs a backend database (e.g., Firebase) and authentication; out of scope for a single‑plot prototype. |
| **SMS fallback** – Send plain‑text SMS instead of Telegram/WhatsApp. | Twilio costs become non‑zero after trial; can be added later if user base lacks internet messaging. |
| **Advanced weather inputs** – Solar radiation, wind speed, soil temperature. | Sensors increase BOM cost; basic temperature/humidity + forecast is adequate for initial recommendation. |

---

### Quick‑Start Checklist (for the student team)

1. **Create accounts** – Google Cloud (service account), OpenWeatherMap, Telegram BotFather.  
2. **Assemble hardware** – Wire sensor to ESP32, flash sketch that POSTs JSON to a Google Apps Script endpoint (or directly to Sheets via API).  
3. **Test data flow** – Verify a row appears in the Sheet within 10 s of a sensor reading.  
4. **Clone repo** – Open the Colab notebook, install required pip packages (`!pip install gspread pandas matplotlib plotly fpdf

In [ ]:

import time

# STEP 1: Critic reviews the current execution plan.
critic_objective = """
Review the proposed execution roadmap critically.

Check:
1. Are the steps clear, ordered, and actionable?
2. Are the time estimates realistic for the stated scope?
3. Are the required tools and dependencies identified?
4. Are testing criteria measurable?
5. Are risks and mitigations included?
6. Are assumptions and uncertain figures clearly labelled?
7. Is the plan suitable for a student prototype?
8. Are hardware deployment and notebook simulation clearly separated?

Start your response with EXACTLY one of these lines:
VERDICT: PASS
VERDICT: REVISE

Then list the strongest parts, problems found, and specific improvements.
Do not invent facts or request unnecessary advanced features.
Remember: this roadmap is for the submitted example idea, not the
AI Idea-to-Execution Engine itself.
Keep the review concise.
"""

critic_review = run_ai_role(
    role="Critical Plan Reviewer",
    objective=critic_objective,
    context=(
        f"SUBMITTED IDEA:\n{sample_idea}\n\n"
        f"EXECUTION ROADMAP:\n{execution_plan[:5000]}"
    )
)

display(Markdown("## 4. Critic Review\n\n" + critic_review))

# STEP 2: Revise only if the critic requests changes.
revision_log = []
current_plan = execution_plan

first_line = critic_review.strip().splitlines()[0].upper()

if "VERDICT: REVISE" in first_line:
    for attempt in range(1, 3):
        print(f"\nRevision attempt {attempt} of 2...")
        time.sleep(12)

        try:
            revised_plan = run_ai_role(
                role="Execution Plan Improvement Agent",
                objective=(
                    "Improve the existing execution roadmap using the "
                    "critic's feedback. Preserve useful content, fix the "
                    "identified problems, and keep the plan realistic. "
                    "Label assumptions and estimates. Return the complete "
                    "improved roadmap in under 800 words."
                ),
                context=(
                    f"IDEA:\n{sample_idea}\n\n"
                    f"CURRENT PLAN:\n{current_plan[:3500]}\n\n"
                    f"CRITIC FEEDBACK:\n{critic_review[:1800]}"
                )
            )

            current_plan = revised_plan
            revision_log.append(
                f"Revision {attempt}: Plan updated using critic feedback."
            )

            # Re-review the revised plan.
            time.sleep(12)

            critic_review = run_ai_role(
                role="Critical Plan Reviewer",
                objective=critic_objective,
                context=(
                    f"IDEA:\n{sample_idea}\n\n"
                    f"REVISED ROADMAP:\n{current_plan[:4500]}"
                )
            )

            display(
                Markdown(
                    f"## Critic Review After Revision {attempt}\n\n"
                    + critic_review
                )
            )

            first_line = critic_review.strip().splitlines()[0].upper()

            if "VERDICT: PASS" in first_line:
                print("Critic accepted the revised plan.")
                break

            if "VERDICT: REVISE" not in first_line:
                print("Verdict format unclear; stopping further revisions.")
                break

        except Exception as e:
            print("Revision stopped:", type(e).__name__, str(e))
            print("The last successfully generated plan is preserved.")
            break

else:
    print("The critic did not request a revision.")
    revision_log.append("Initial plan reviewed; no revision requested.")

# Keep the final plan available for the report.
execution_plan = current_plan

print("\nCritic-and-revision workflow finished.")
print("Revision attempts completed:", len(revision_log))
print("Project title: AI Idea-to-Execution Engine")


Running AI role: Critical Plan Reviewer...
Completed: Critical Plan Reviewer


## 4. Critic Review

VERDICT: REVISE  

**Strongest Parts**  
- **Clear, ordered steps** – The roadmap follows a logical progression from requirements to demo, with numbered tasks and deliverables.  
- **Realistic tooling** – All hardware and cloud services are low‑cost/free and readily available to students; the dependency list is complete.  
- **Measurable testing criteria** – Data‑integrity, rule‑engine accuracy, and message‑delivery latency are quantified with concrete thresholds.  
- **Risk awareness** – Connectivity and threshold‑selection risks are identified, and basic mitigations are proposed.  

**Problems Found**  
1. **Incomplete risk & mitigation table** – The “Threshold selection uncertainty” row is cut off, leaving the mitigation unfinished.  
2. **Assumptions not flagged** – Key assumptions (e.g., Wi‑Fi availability, sensor calibration tolerance, weather‑API rate limits) are embedded in the text but not labeled as such.  
3. **Hardware vs. notebook separation** – While the sensor node and cloud pipeline are described, the plan does not explicitly state that the Colab notebook will be run *offline* for simulation or that hardware deployment can be swapped for synthetic data during early development.  
4. **Time‑estimate optimism** – 1 week for “Build & test sensor node” and “Set up cloud data pipeline” may be tight for a 2‑3‑person team with limited prior experience; no buffer for debugging is shown.  
5. **Testing scope limited** – The rule‑engine accuracy test relies on “manually‑labelled” days but does not specify how those labels will be obtained (e.g., ground‑truth moisture probe).  
6. **Documentation depth** – The user‑guide is mentioned but not broken into sections (setup, calibration, troubleshooting), which could affect reproducibility.  

**Specific Improvements**  

| Area | Recommendation |
|------|----------------|
| **Complete Risks** | Finish the “Threshold selection uncertainty” row (e.g., mitigation: start with literature values, allow farmer to adjust thresholds via a simple config sheet, and log outcomes for iterative tuning). Add a third risk such as “Google Sheets API quota exhaustion” with mitigation (cache data locally, limit read frequency). |
| **Label Assumptions** | Add an “Assumptions” block (e.g., <br>• Wi‑Fi available at least 4 h/day <br>• Capacitive sensor accuracy ±5 % after calibration <br>• OpenWeatherMap free tier provides ≤60 calls/day). Clearly mark each with “Assumption”. |
| **Separate Simulation & Deployment** | Insert a step before #2: “Simulate sensor data in Colab using synthetic CSV” (≈0.5 day). State that the notebook can be run with either live Google‑Sheet data or a local CSV, enabling development without hardware. |
| **Add Buffer Time** | Increase the duration for steps #2 and #3 by 0.5 week each (or add a “Contingency” 0.5 week at the end) to accommodate debugging and connectivity issues. |
| **Define Ground‑Truth Collection** | In the testing criteria, specify that a calibrated reference moisture probe will be used on the test plot to label “water‑needed” days, and that at least 10 labeled days will be collected before evaluating the 80 % accuracy target. |
| **Expand Documentation** | Break the user guide into: <br>1. Hardware assembly <br>2. Cloud‑service setup (API keys, Sheet sharing) <br>3. Threshold calibration <br>4. Running the notebook <br>5. Interpreting the recommendation <br>6. Troubleshooting (connectivity, missing data). |
| **Clarify Deliverables** | For each step, list the artifact (e.g., “Step 3: `gsheet_helper.py` with unit tests”) to make the output verifiable. |
| **Add Reproducibility Checklist** | At the end of the repo, include a checklist: “Can a new student clone the repo, flash the ESP32, and run the notebook without external help? – Yes/No”. This ensures the prototype meets the “student‑level” requirement. |

Implementing these refinements will make the roadmap fully actionable, transparent about uncertainties, and better suited for a student prototype that can be demonstrated reliably in Google Colab.


Revision attempt 1 of 2...

Running AI role: Execution Plan Improvement Agent...
Completed: Execution Plan Improvement Agent

Running AI role: Critical Plan Reviewer...
Completed: Critical Plan Reviewer


## Critic Review After Revision 1

VERDICT: REVISE  

---

## Strongest Parts  

| # | What works well |
|---|-----------------|
| **Assumptions** | All key external dependencies (Wi‑Fi, sensor tolerance, API limits, Telegram, Google Sheets) are explicitly listed and flagged in bold. |
| **Risk‑Mitigation Table** | Risks are identified, impact/likelihood are rated, and concrete mitigations (local SD fallback, weekly calibration, caching, alternative email) are provided. |
| **Parallel Work & Buffers** | The timeline recognises parallel development (hardware vs. notebook) and adds explicit buffer hours for each phase. |
| **Student‑Friendly Scope** | Uses inexpensive ESP32 kit, free APIs, Google Sheets, and a Colab notebook – all readily accessible for a university project. |
| **Field‑Trial Design** | Includes a short (3‑5 day) on‑site test, ground‑truth moisture measurement, and farmer‑feedback loop. |

---

## Problems Found  

| # | Issue | Why it matters |
|---|-------|----------------|
| **1. Actionability of Steps** | Some activities are vague (“finalize thresholds”, “draft notebook skeleton”, “rule‑engine”) without concrete sub‑tasks or acceptance criteria. |
| **2. Time Estimates** | Total effort ≈ 124 h (≈ 15 working days). For a 2‑3‑person part‑time team (≈ 10 h/week each) this stretches to 6‑8 weeks, yet the timeline compresses everything into 11 calendar weeks with only 1‑2 h/week per person in many weeks. The “Week 2‑3” block (20 h) assumes 10 h/week per person, which may be unrealistic given coursework. |
| **3. Missing Tool/Dependency Details** | The roadmap mentions `gspread`, Telegram Bot API, and OpenWeatherMap but does not list required Python packages, version constraints, or hardware libraries (e.g., `ArduinoJson`, `ESPAsyncWebServer`). No mention of Colab runtime limits (e.g., file‑size, network restrictions). |
| **4. Testing Criteria Not Quantified** | The only metric is “simple accuracy metric (recommendation = watered when probe < 30 % VWC)”. No thresholds for acceptable precision/recall, no definition of false‑positive/negative costs, and no statistical validation plan. |
| **5. Risks Not Exhaustive** | No explicit risk for **power supply** (battery depletion), **sensor failure**, or **data privacy** (Telegram bot token exposure). |
| **6. Assumptions Not Clearly Marked as Uncertain** | While bolded, they are not separated into a dedicated “Uncertain Figures” section, making it harder for reviewers to spot which numbers are estimates. |
| **7. Hardware vs. Notebook Separation** | The plan mixes hardware tasks (sensor node) with notebook tasks in the same weekly blocks (e.g., Week 5‑6). It is not clear when the hardware is fully functional before notebook integration begins. |
| **8. Documentation Scope** | Only a generic README is mentioned; no plan for **code repository structure**, **environment reproducibility** (requirements.txt / environment.yml), or **video demo** for the prototype showcase. |
| **9. No Clear Success Metrics** | The roadmap says “define success metrics” in Week 0 but never revisits them. Without measurable goals (e.g., “≥ 80 % recommendation accuracy”) the evaluation in Week 10 lacks a benchmark. |

---

## Specific Improvements  

1. **Break Down High‑Level Activities into Sub‑tasks**  
   *Example*: “Finalize thresholds” → (a) research crop‑specific VWC ranges, (b) create a threshold matrix in a spreadsheet, (c) select initial default (e.g., 30 % VWC).  
   Add a **Definition of Done** for each week (e.g., “ESP32 firmware uploads and prints JSON payload to serial monitor”).  

2. **Adjust Time Estimates & Add Weekly Capacity Planning**  
   - Assume each student can commit **8 h/week** (typical part‑time). Re‑calculate effort per week to fit within 11 weeks or extend the schedule to **14‑15 weeks**.  
   - Insert a **Gantt‑style view** with overlapping tasks clearly labeled (hardware ready → notebook integration).  

3. **Explicit Tool & Dependency List**  
   - Hardware: ESP32‑DevKitC, Capacitive Soil Moisture Sensor, micro‑SD card module, 3.7 V Li‑Po battery + charger, jumper wires, enclosure.  
   - Firmware libraries: `ArduinoJson`, `WiFi.h`, `SD.h`.  
   - Colab/Python: `gspread`, `oauth2client`, `requests`, `pandas`, `matplotlib`, `fpdf`, `python‑telegram‑bot`.  
   - Mention Colab limitations (no persistent local storage, need to mount Google Drive for PDFs).  

4. **Define Measurable Testing Criteria**  
   - **Accuracy**: % of recommendations that match ground‑truth watering decision (target ≥ 80 %).  
   - **Latency**: end‑to‑end time from sensor reading to Telegram message ≤ 30 s (average).  
   - **Uptime**: ≥ 95 % of expected sensor uploads over the trial period.  
   - **User Satisfaction**: simple Likert‑scale survey (≥ 4/5).  

5. **Add Missing Risks & Mitigations**  
   - **Power**: Battery life < 48 h → use solar panel or schedule deep‑sleep between uploads.  
   - **Sensor Failure**: Include self‑test routine that sends “sensor error” flag to Sheet.  
   - **Security**: Store API keys in Google Cloud Secret Manager or Colab environment variables; never hard‑code in notebook.  

6. **Separate Hardware Completion Milestone**  
   - Insert a **“Hardware Ready”** milestone at end of Week 3. Only after this point should notebook developers start pulling real data (Week 4 onward).  

7. **Clarify Assumptions Section**  
   - Move all bolded assumptions into a dedicated “Uncertain Figures” table with a column for **Confidence (Low/Medium/High)** and **Verification Plan** (e.g., “Test Wi‑Fi availability on site for 2 days”).  

8. **Expand Documentation Deliverables**  
   - `README.md` with quick‑start, hardware wiring diagram, and troubleshooting FAQ.  
   - `requirements.txt` and a Colab “Setup” cell that installs exact versions.  
   - A short **demo video (≤ 5 min)** showing sensor data flow to Telegram.  

9. **Re‑introduce Success Metrics in Week 0 and Re‑visit in Week 10**  
   - List concrete KPIs (accuracy, latency, uptime, user rating) in the kickoff and use them as evaluation criteria in the final week.  

---

Implementing the above refinements will make the roadmap **clearer, more measurable, and better aligned with a realistic student workload**, while preserving the low‑budget, accessible nature of the prototype. Once the revisions are incorporated, the plan should meet all review criteria.


Revision attempt 2 of 2...

Running AI role: Execution Plan Improvement Agent...
Completed: Execution Plan Improvement Agent

Running AI role: Critical Plan Reviewer...
Completed: Critical Plan Reviewer


## Critic Review After Revision 2

VERDICT: REVISE  

---

## Strongest Parts  

1. **Explicit Assumptions & Risk Table** – Clear, bold‑faced assumptions (A1‑A7) and a risk‑mitigation matrix give the team early warning signals.  
2. **Structured Phases & Kanban Setup** – Phase 0 defines concrete deliverables (repo, CI, metrics) that are immediately actionable for a student team.  
3. **Tool List & Dependency Capture** – Arduino IDE, ArduinoJson, OpenWeatherMap, Telegram Bot, Google Sheets API, GitHub Actions, and Colab are all identified, with notes on where to obtain credentials.  
4. **Buffers & Slack** – Each phase includes a 1‑day or 2‑hour buffer, which is realistic for a part‑time student schedule.  

---

## Problems Found  

| # | Issue | Why it matters |
|---|-------|----------------|
| **P1** | **Time estimates are optimistic for hardware prototyping** – 30 h for weeks 1‑3 assumes 8 h/week per person, but ordering, soldering, debugging, and calibration often consume more time, especially for novices. |
| **P2** | **Testing criteria are vague** – Success metrics are mentioned only in Phase 0 (“≤ 10 % VWC error, ≤ 5 min alert latency”) but no concrete test plan, data‑set size, or acceptance thresholds for each sub‑task. |
| **P3** | **Missing separation of “hardware deployment” vs “notebook simulation”** – The roadmap blends ESP32 code development with Colab pipeline without a dedicated “simulation‑only” sprint to prove the data flow before field hardware is installed. |
| **P4** | **Assumptions not consistently flagged as uncertain** – A6 (Google Sheets quota) and A7 (Colab runtime) are presented as facts; they should be marked with “*uncertain*” and a verification step. |
| **P5** | **Tool versioning & reproducibility not fully captured** – Only Arduino IDE version is listed; Python package versions, Colab notebook dependencies, and hardware firmware versions are omitted, which can cause reproducibility problems for a student hand‑off. |
| **P6** | **No explicit validation of the irrigation recommendation algorithm** – The roadmap states “collect farmer feedback” but does not define how to evaluate recommendation quality (e.g., confusion matrix, water‑saved metric). |
| **P7** | **Risk R4 (optimal moisture threshold) lacks a mitigation schedule** – It suggests “collect feedback each day” but does not allocate time or define a decision point for threshold adjustment. |
| **P8** | **Limited detail on data‑pipeline testing** – No unit‑test or integration‑test plan for JSON upload, Google Sheets write, or Telegram message flow. |
| **P9** | **Budget considerations are absent** – The “low‑budget” requirement is not quantified; component costs, API usage costs, or Colab compute limits are not tracked. |

---

## Specific Improvements  

1. **Refine Time Estimates**  
   * Add a 25 % contingency to hardware weeks (≈ 38 h total).  
   * Split Phase 1 into two sub‑phases: “Prototype on Breadboard” (≈ 12 h) and “Robust Enclosure & Field‑Ready Build” (≈ 16 h).  

2. **Add a Dedicated Simulation Sprint**  
   * **Phase 0‑5 (Week 0‑1, 6 h)** – Build a Colab notebook that **simulates sensor data** (synthetic VWC series, weather API mock).  
   * Validate end‑to‑end pipeline (JSON → Sheets → Telegram) without any hardware.  
   * Acceptance: 100 % of simulated messages appear in the sheet and bot within 2 min.  

3. **Formalize Testing Criteria**  
   * **Unit Tests** – Arduino sketch: verify ADC → VWC conversion using known voltage values (use `assert` in the sketch).  
   * **Integration Tests** – Python: mock HTTP POST, assert correct JSON schema, verify Google Sheets write count.  
   * **System Tests** – Deploy sensor for 48 h, compare logged VWC against a calibrated reference probe; target MAE ≤ 5 % VWC.  
   * Document all tests in a `README.md` with pass/fail thresholds.  

4. **Mark Uncertain Assumptions & Add Verification Steps**  
   * Re‑label A6 and A7 as **“Assumption (to be verified)”**.  
   * Insert a “Verification” task at the end of Phase 0: run a quick Colab cell that writes 600 rows to the test sheet and makes 1 200 API calls to confirm quotas.  

5. **Capture Full Toolchain Versions**  
   * Add a `requirements.txt` for Python (e.g., `pandas==2.2.0`, `gspread==5.9.0`, `requests==2.31.0`).  
   * Record ESP32 core version (e.g., `esp32-arduino@2.0.9`).  
   * Include a `environment.yml` for Colab reproducibility.  

6. **Define Irrigation Recommendation Validation**  
   * Use a simple rule‑based model (VWC < threshold → “Water”).  
   * After the 1‑week field trial, compute **Precision** (correct water alerts / total alerts) and **Recall** (correct water alerts / total needed water events) using farmer’s manual watering log.  
   * Target Precision ≥ 80 % and Recall ≥ 70 %.  

7. **Schedule Threshold Adjustment (Risk R4)**  
   * Add a **“Threshold Review”** meeting at the end of Week 4, with a 2‑hour slot to analyze feedback and update the notebook constant.  

8. **Include Budget Table**  
   * List component costs (ESP32 ≈ $8, sensor ≈ $5, SD‑card ≈ $3, wiring ≈ $2).  
   * Add API cost placeholders (OpenWeatherMap free tier, Telegram free, Google Sheets free).  
   * Total projected spend ≈ $20, well under a typical student grant.  

9. **Separate Hardware Deployment Deliverables**  
   * **Hardware Deliverable** – “Field‑Ready ESP32 Unit” with enclosure, SD‑card logging, and Wi‑Fi antenna.  
   * **Software Deliverable** – “Colab Data Pipeline Notebook” with simulated and live data modes.  
   * Clearly state hand‑off point: after Phase 2, the hardware unit will push real JSON to the same endpoint used by the notebook.  

10. **Add a Final Demo Checklist**  
    * Live sensor streaming to Google Sheet for ≥ 30 min.  
    * Telegram alert received within 3 min of threshold breach.  
    * Notebook reproduces the same recommendation using recorded data.  
    * All source code, docs, and CI pipelines pass on a fresh GitHub clone.  

---

### Summary  

The roadmap is well‑structured and shows good awareness of risks, but it needs clearer separation between simulation and hardware, more concrete testing metrics, realistic time buffers, and explicit reproducibility details. Incorporating the improvements above will make the plan robust enough for a student prototype while keeping the low‑budget, Colab‑friendly constraints.


Critic-and-revision workflow finished.
Revision attempts completed: 2
Project title: AI Idea-to-Execution Engine


In [ ]:

# AI Idea-to-Execution Engine
# Accept a different idea every time you run this cell.

user_idea = input(
    "Enter any idea you want to turn into an execution plan:\n> "
).strip()

if not user_idea:
    print("Please enter an idea and run this cell again.")
else:
    sample_idea = user_idea  # Keeps compatibility with existing workflow cells.
    workflow_history = []

    print("\nProject: AI Idea-to-Execution Engine")
    print("Your idea has been received!")
    print("\nIdea:")
    print(user_idea)

Enter any idea you want to turn into an execution plan:
> Create a system to reduce food waste

Project: AI Idea-to-Execution Engine
Your idea has been received!

Idea:
Create a system to reduce food waste


In [ ]:

# ============================================================
# AI IDEA-TO-EXECUTION ENGINE
# One cell | Any input | Multiple ideas | Type exit to stop
# ============================================================

import time
from IPython.display import display, Markdown

# Check existing Groq setup
if "client" not in globals() or "GROQ_MODEL" not in globals():
    raise ValueError("Run your Groq API setup cell first.")

def ask_ai(role, task, context, max_tokens=700):
    """Call the configured Groq model with a bounded retry."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are part of the AI Idea-to-Execution Engine. "
                "Adapt to ANY domain or input: business, technology, "
                "education, personal goals, social problems, creative work, "
                "operations, research, or other tasks. Never assume a fixed "
                "topic. Be practical and concise. Clearly label estimates "
                "and assumptions. Never claim actions were actually executed."
            )
        },
        {
            "role": "user",
            "content": (
                f"USER INPUT:\n{context}\n\n"
                f"YOUR ROLE: {role}\n\nTASK:\n{task}"
            )
        }
    ]

    for attempt in range(2):
        try:
            result = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                temperature=0.2,
                max_tokens=max_tokens
            ).choices[0].message.content

            if not result or not result.strip():
                raise ValueError("The model returned an empty response.")

            return result.strip()

        except Exception as e:
            error = str(e).lower()
            if attempt == 0 and (
                "429" in error or "rate_limit" in error
                or "rate limit" in error
            ):
                print("Rate limit reached; waiting 15 seconds...")
                time.sleep(15)
            else:
                raise


def execute_idea(user_input):
    """Run the complete workflow for one submitted idea."""

    history = []
    print("\n" + "=" * 65)
    print("AI IDEA-TO-EXECUTION ENGINE")
    print("Input:", user_input)
    print("=" * 65)

    # Retrieve relevant RAG knowledge if the existing function is available.
    knowledge = ""
    if "retrieve_documents" in globals():
        try:
            knowledge = str(retrieve_documents(user_input, top_k=3))[:1200]
        except Exception:
            pass

    if not knowledge:
        knowledge = "No retrieved notes. Use general knowledge carefully."

    # Stage 1: Analyze any type of input.
    print("\n[1/4] Analyzing input...")
    analysis = ask_ai(
        "Universal Input Analyst",
        """
        Identify the user's intended outcome and the type of request.
        Analyze the problem or goal, intended users where relevant,
        requirements, constraints, possible approaches, and first milestone.
        If the user asks a direct question, answer it directly. If the input
        is vague, state assumptions and ask essential clarifying questions
        while still giving a useful initial response. Do not force a project
        template on a simple question or non-technical goal.
        """,
        f"USER INPUT:\n{user_input}\n\nKNOWLEDGE:\n{knowledge}",
        650
    )
    history.append(("Input Analysis", analysis))
    display(Markdown("## 1. Input Analysis\n\n" + analysis))

    # Stage 2: Evaluate feasibility.
    print("\n[2/4] Evaluating feasibility and risks...")
    feasibility = ask_ai(
        "Feasibility and Risk Evaluator",
        """
        Evaluate the approach for this particular input. Cover relevant
        resources, skills, tools, dependencies, complexity, time or budget
        estimates where useful, major risks and mitigations, and ways to
        measure success. Adapt to the domain. Label estimates and assumptions.
        Do not invent verified prices, statistics, or guaranteed outcomes.
        """,
        f"USER INPUT:\n{user_input}\n\nANALYSIS:\n{analysis}",
        650
    )
    history.append(("Feasibility Evaluation", feasibility))
    display(Markdown("## 2. Feasibility Evaluation\n\n" + feasibility))

    # Stage 3: Create a customized execution plan.
    print("\n[3/4] Creating execution plan...")
    plan = ask_ai(
        "Adaptive Execution Planner",
        """
        Create a customized plan for the original user input. Choose steps
        appropriate to its actual domain; do not use a fixed topic-specific
        template. Include the desired outcome, prioritized sequential steps,
        deliverables, relevant resources and dependencies, time or budget
        estimates where useful, measurable success criteria, risks and
        mitigations, and the immediate next action. For simple questions,
        give a proportionate answer and useful next steps. For vague inputs,
        state assumptions and provide a provisional plan. Keep the response
        under approximately 650 words. Do not claim the plan was executed.
        """,
        f"USER INPUT:\n{user_input}\n\n"
        f"ANALYSIS:\n{analysis[:1800]}\n\n"
        f"FEASIBILITY:\n{feasibility[:1600]}",
        950
    )
    history.append(("Execution Plan", plan))
    display(Markdown("## 3. Customized Execution Plan\n\n" + plan))

    # Stage 4: Independent critic review.
    print("\n[4/4] Reviewing plan...")
    review = ask_ai(
        "Independent Plan Critic",
        """
        Check whether the plan addresses the original input, is practical,
        has clear priorities and deliverables, includes relevant success
        criteria, handles important risks, and labels uncertainty. Adapt
        your review to the domain and do not demand irrelevant features.
        Begin with exactly 'VERDICT: PASS' or 'VERDICT: REVISE'.
        Then give concise feedback.
        """,
        f"ORIGINAL INPUT:\n{user_input}\n\nPLAN:\n{plan[:3500]}",
        400
    )
    history.append(("Critic Review", review))
    display(Markdown("## 4. Critic Review\n\n" + review))

    # Revise at most once.
    revision_status = "No revision requested."
    if review.upper().startswith("VERDICT: REVISE"):
        print("\nImproving the plan based on critic feedback...")
        try:
            improved_plan = ask_ai(
                "Plan Improvement Agent",
                """
                Improve the plan using the critic's feedback. Preserve useful
                content, fix relevant weaknesses, and tailor the result to the
                original input. Avoid unnecessary complexity. Label estimates
                and assumptions. Return the complete improved plan.
                """,
                f"ORIGINAL INPUT:\n{user_input}\n\n"
                f"CURRENT PLAN:\n{plan[:2800]}\n\n"
                f"CRITIC FEEDBACK:\n{review[:900]}",
                950
            )
            plan = improved_plan
            revision_status = "Plan revised using critic feedback."
            history.append(("Improved Plan", plan))
            display(Markdown("## 5. Improved Execution Plan\n\n" + plan))
        except Exception as e:
            revision_status = "Revision failed; original plan retained."
            print("Revision unavailable:", type(e).__name__, str(e))

    # Build a report for this particular input.
    report = f"""# AI Idea-to-Execution Engine

## User Input
{user_input}

## 1. Input Analysis
{analysis}

## 2. Feasibility Evaluation
{feasibility}

## 3. Final Execution Plan
{plan}

## 4. Critic Review
{review}

## 5. Review Status
{revision_status}

## Limitations
This is AI-generated guidance. Estimates and assumptions may require
verification. Generating a plan does not mean external actions have
been performed or that the outcome has been achieved.
"""

    # Save a separate report for each input, without extra dependencies.
    import os
    os.makedirs("/content/idea_execution_reports", exist_ok=True)
    safe_name = "".join(
        ch if ch.isalnum() else "_"
        for ch in user_input[:45]
    ).strip("_") or "idea"
    filename = f"/content/idea_execution_reports/{safe_name}.md"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(report)

    print("\nReport saved:", filename)
    print("AI stages completed:", len(history))
    return report


# ============================================================
# CONTINUOUS INPUT LOOP
# Enter any number of ideas. Type exit to stop.
# ============================================================

print("\n" + "=" * 65)
print("AI IDEA-TO-EXECUTION ENGINE IS READY")
print("Enter any idea, goal, problem, or task.")
print("Type exit, quit, or stop to finish.")
print("=" * 65)

while True:
    try:
        user_input = input("\nEnter your idea or task:\n> ").strip()

        if user_input.lower() in ("exit", "quit", "stop"):
            print("\nEngine stopped. Your generated reports remain in:")
            print("/content/idea_execution_reports")
            break

        if not user_input:
            print("Empty input. Please enter an idea or type 'exit'.")
            continue

        try:
            execute_idea(user_input)
        except Exception as e:
            print("\nThis input could not be completed.")
            print("Error:", type(e).__name__, str(e))
            print("You can try another input without restarting the engine.")

    except KeyboardInterrupt:
        print("\nEngine stopped by user.")
        break
    except EOFError:
        print("\nInput session ended.")
        break


AI IDEA-TO-EXECUTION ENGINE IS READY
Enter any idea, goal, problem, or task.
Type exit, quit, or stop to finish.

Enter your idea or task:
> Start a small bakery

AI IDEA-TO-EXECUTION ENGINE
Input: Start a small bakery

[1/4] Analyzing input...


## 1. Input Analysis

**Intended Outcome**  
You want to launch a *small bakery* – a physical (or possibly hybrid) food‑service business that produces and sells baked goods.

**Type of Request**  
This is a *project‑planning / idea‑analysis* request. You’re looking for an initial breakdown of what’s needed to get the bakery off the ground, not a single factual answer.

---

## 1. Quick Problem / Goal Statement  
**Goal:** Open a profitable, locally‑focused bakery that serves a defined customer segment with a core menu of high‑quality baked goods.  
**Success Metric (first 6 months):** Reach break‑even or generate a modest profit (e.g., ≥ 10 % margin) while maintaining consistent product quality and customer satisfaction (e.g., ≥ 4‑star rating on local review sites).

---

## 2. Core Assumptions (feel free to adjust)

| # | Assumption | Why it matters |
|---|------------|----------------|
| A1 | You have or can secure **$30‑$60 k** of startup capital (equipment, lease, permits, initial inventory). | Determines scale of equipment, location size, and marketing spend. |
| A2 | The bakery will be **brick‑and‑mortar** with a modest footprint (≈ 500‑800 sq ft). | Affects lease cost, layout, and staffing needs. |
| A3 | Target market is **local residents & nearby office workers** within a 2‑mile radius. | Guides product mix, pricing, and opening hours. |
| A4 | You will start with a **minimum viable product (MVP) menu** (e.g., 3 breads, 3 pastries, 1 cake). | Keeps inventory, waste, and complexity low while you test demand. |
| A5 | You have (or can acquire) basic **baking skills** or will hire a qualified baker. | Critical for product quality and regulatory compliance. |
| A6 | Local health‑department regulations require a **food‑service permit** and periodic inspections. | Must be budgeted for time and fees. |

If any of these don’t fit your situation, let me know and we’ll adjust the plan.

---

## 3. Intended Users / Customers  

| Segment | Characteristics | Primary Needs |
|---------|----------------|---------------|
| **Neighborhood families** | 30‑60 % of foot traffic, value fresh, affordable treats for kids & gatherings. | Consistent quality, kid‑friendly options, reasonable price. |
| **Office workers / commuters** | 20‑30 % of traffic, limited time, seek quick grab‑and‑go items. | Speed of service, portable items (cookies, muffins), coffee pairing. |
| **Local food‑enthusiasts** | Smaller but vocal


[2/4] Evaluating feasibility and risks...


## 2. Feasibility Evaluation

## Feasibility & Risk Evaluation – “Start a Small Bakery”

---

### 1. Core Assumptions (you can tweak)

| # | Assumption | Impact on Plan |
|---|------------|----------------|
| **A1** | **Startup capital ≈ $30‑$60 k** (covers lease, equipment, permits, initial inventory, modest marketing). | Sets ceiling for equipment size, location rent, staffing level, and cash‑flow buffer. |
| **A2** | **Brick‑and‑mortar shop** of **500‑800 sq ft** in a mixed‑residential/commercial zone. | Determines lease cost, utility load, and layout requirements. |
| **A3** | **Target market** = local residents + nearby office workers (≈ 2‑mile radius). | Drives product mix, pricing, opening hours, and promotional channels. |
| **A4** | **MVP menu** – 3 breads, 3 pastries, 1 cake (≈ 10 SKUs). | Keeps inventory, waste, and labor low while you test demand. |
| **A5** | **Baking competence** – you can bake or will hire a qualified baker (≥ 2 years experience). | Directly affects product quality, consistency, and regulatory compliance. |
| **A6** | **Regulatory environment** – local health department requires food‑service permit, fire‑safety clearance, and annual inspections. | Must be budgeted for fees, time, and possible design changes. |

> **If any assumption is off, adjust the numbers below accordingly.**

---

### 2. Required Resources & Tools

| Category | Specific Items | Typical Cost / Time* | Notes |
|----------|----------------|----------------------|-------|
| **Physical Space** | Lease (500‑800 sq ft), utilities, basic build‑out (kitchen hood, plumbing). | **$1,500‑$3,000 / mo** rent; **$5‑$15 k** build‑out (depends on existing kitchen). | Look for “restaurant‑ready” spaces to reduce build‑out. |
| **Equipment** | - Commercial convection oven (30‑40 qt) <br> - Deck oven or countertop oven (optional) <br> - Mixer (10‑15 qt) <br> - Proofing cabinet <br> - Refrigeration (reach‑in cooler, display case) <br> - Small prep tables, shelving, pans, mixers, scales, dough sheeters. | **$15‑$30 k** total (used equipment can cut cost 30‑40%). | Prioritize


[3/4] Creating execution plan...


...


[3/4] Creating execution plan...


## 3. Customized Execution Plan

**Goal**  
Enroll in and successfully complete a four‑year Bachelor of Technology (B‑Tech) degree that matches your interests and career aspirations.

---

## 1. Core Assumptions  *(adjust if any differ)*  

| # | Assumption |
|---|------------|
| **A1** | You are currently in the final year of 10+2 (or have already completed it) with Physics, Chemistry and Mathematics (or Biology + Math). |
| **A2** | You are based in India or a country with a similar entrance‑exam‑driven admission system. |
| **A3** | You aim for a full‑time, on‑campus B‑Tech (not a diploma, distance‑learning, or overseas program). |
| **A4** | You can devote 6‑12 months to entrance‑exam preparation and have a budget of **₹ 2 – 10 Lakh** (or equivalent) for tuition, fees and living costs, or you can obtain a loan/scholarship. |
| **A5** | You are open to either a government‑aided institute (NIT/IIIT/IIT, state colleges) or a reputable private college. |

If any of these do not hold, see the “Alternative Paths” box at the end.

---

## 2. Success Criteria  

| Metric | Target |
|--------|--------|
| **Eligibility** | 12‑th aggregate ≥ 50 % with required subjects. |
| **Entrance‑Exam Rank** | Within the cutoff for at least one preferred college/branch (e.g., JEE Main rank ≤ 150,000 for NITs; state‑exam rank ≤ 10,000 for top state colleges). |
| **Admission Offer** | Formal seat allocation and fee‑payment confirmation. |
| **Graduation** | Minimum CGPA ≥ 6.0 (or equivalent) and successful completion of all 8 semesters. |
| **Placement** | At least one on‑campus offer with a package ≥ ₹ 4 LPA (or local market equivalent) by final year. |

---

## 3. Prioritized Sequential Steps  

| Phase | Action | Deliverable | Time Estimate* | Dependencies |
|-------|--------|-------------|----------------|--------------|
| **1️⃣ Eligibility Check** | Collect 12‑th mark sheets, verify subject mix, calculate aggregate. | Eligibility checklist (✓/✗) | 1 day | None |
| **2️⃣ Career & Branch Selection** | • Take a short interest‑skill quiz.<br>• Review industry growth (e.g., NASSCOM, CII).<br>• Shortlist 3‑4 streams (e.g., CS, ECE, Mechanical). | Ranked list of preferred branches with rationale. | 1 week | Phase 1 |
| **3️⃣ Entrance‑Exam Roadmap** | Choose primary exam(s): JEE Main → JEE Advanced (IIT route) or State CET/Private (BITSAT, SRMJEE). | Exam‑selection matrix (exam, fee, date, cutoff). | 2 days | Phase 2 |
| **4️⃣ Study Plan & Resources** | • Draft a 30‑week syllabus calendar (concept → practice → mock).<br>• Gather core books (NCERT, H.C. Verma, O.P. Tandon).<br>• Enroll in free/paid online courses (NPTEL, Unacademy).<br>• If budget permits, register for a coaching centre (optional). | Detailed weekly study schedule + resource list. | 1 week | Phase 3 |
| **5️⃣ Mock‑Test & Performance Tracking** | • Take a full‑length mock every 2 weeks.<br>• Record scores, identify weak topics, adjust plan. | Performance dashboard (score trend, target‑gap). | Ongoing (6‑10 months) | Phase 4 |
| **6️⃣ Exam Registration & Logistics** | • Pay exam fee, upload documents, note centre & date.<br>• Arrange travel/accommodation for


[4/4] Reviewing plan...


## 4. Critic Review

**VERDICT: REVISE**

**Feedback**

1. **Alignment with Goal** – The plan correctly targets enrollment and completion of a B‑Tech, but it stops at “Exam Registration & Logistics” and does not cover post‑admission steps (college onboarding, semester planning, internship/placement preparation). Add phases for **college enrollment**, **academic progression**, and **career/placement strategy** to ensure the “successful completion” objective is fully addressed.

2. **Risk Management** – Risks such as low entrance‑exam rank, financial constraints, health/personal emergencies, and COVID‑related disruptions are not explicitly identified or mitigated. Include a **Risk Register** with mitigation actions (e.g., backup colleges, scholarship search, contingency study schedule).

3. **Uncertainty Labels** – The plan lists assumptions but does not flag uncertain items within the steps (e.g., “exam‑selection matrix” depends on future cutoff trends). Use **[UNCERTAIN]** tags where outcomes are not guaranteed.

4. **Deliverables & Success Metrics** – Deliverables are clear for early phases, but later phases lack concrete outputs (e.g., “College enrollment confirmation”, “Semester‑wise GPA tracker”, “Internship offer”). Define these to keep progress measurable.

5. **Time Estimates** – The “Time Estimate*” column uses vague ranges (e.g.,


Report saved: /content/idea_execution_reports/I_want_to_do_B_tech.md
AI stages completed: 4

Enter your idea or task:
> exit

Engine stopped. Your generated reports remain in:
/content/idea_execution_reports
